# 1

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
import numpy as np
from tqdm import tqdm
from collections import Counter
import random
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_fscore_support
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# 2

In [ ]:

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(42)



# 3

In [ ]:

class FixedLandmarkDataset(Dataset):
    def __init__(self, annotations_path, data_root, label_map_path, stats_path,
                 max_frames=70, top_n_classes=200):

        print(f"\nloading dataset from {annotations_path}")
        
        with open(annotations_path, 'r') as f: self.annotations = json.load(f)
        with open(label_map_path, 'r') as f: full_label_map = json.load(f)
        with open(stats_path, 'r') as f: stats = json.load(f)
        
        self.data_root = data_root
        self.max_frames = max_frames
        self.min_frames = 5
        
        self.spatial_dim = 1742
        self.input_dim = self.spatial_dim * 2 
        
        self.mean = torch.tensor(stats['spatial_mean'] + stats['temporal_mean'], dtype=torch.float32)
        self.std = torch.tensor(stats['spatial_std'] + stats['temporal_std'], dtype=torch.float32)
        self.std[self.std < 1e-6] = 1.0 
        print("  loaded global normalization stats.")

        all_glosses = sorted(full_label_map.keys(), key=lambda g: full_label_map[g])
        selected_glosses = all_glosses[:top_n_classes]
        self.gloss_to_idx = {gloss: i for i, gloss in enumerate(selected_glosses)}
        self.idx_to_gloss = {i: gloss for gloss, i in self.gloss_to_idx.items()}
        self.num_classes = len(self.gloss_to_idx)
        
        self.samples = []
        for entry in self.annotations:
            gloss = entry['gloss']
            if gloss not in self.gloss_to_idx: continue
            label_idx = self.gloss_to_idx[gloss]
            for instance in entry.get('instances', []):
                video_id = instance.get('video_id')
                if not video_id: continue
                path = os.path.join(self.data_root, video_id, 'landmarks.json')
                if os.path.exists(path):
                    self.samples.append({'path': path, 'label_idx': label_idx, 'video_id': video_id})
        
        print(f"  found {len(self.samples)} potential samples.")
        self.class_counts = Counter([s['label_idx'] for s in self.samples])

    def __len__(self):
        return len(self.samples)

    def _extract_spatial_features(self, frame):
        if not isinstance(frame, dict): return None
        if 'left_hand_engineered' not in frame or 'right_hand_engineered' not in frame: return None

        def safe_get(key, size):
            data = np.array(frame.get(key, []), dtype=np.float32).flatten()
            if len(data) > size: data = data[:size]
            elif len(data) < size: data = np.pad(data, (0, size - len(data)))
            return data
        
        return np.concatenate([
            safe_get('pose', 132), safe_get('left_hand', 84), safe_get('right_hand', 84),
            safe_get('face', 1404), safe_get('left_hand_engineered', 19), safe_get('right_hand_engineered', 19)
        ])
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        landmarks_path = sample['path']
        
        try:
            with open(landmarks_path, 'r') as f: frames = json.load(f)
            if not isinstance(frames, list) or len(frames) < self.min_frames: return None
        except:
            return None

        spatial_features_list = [self._extract_spatial_features(frame) for frame in frames]
        if any(f is None for f in spatial_features_list): return None

        spatial = np.array(spatial_features_list, dtype=np.float32)
        
        if np.isnan(spatial).any() or np.isinf(spatial).any(): return None
            
        temporal = np.diff(spatial, axis=0, prepend=spatial[0:1])
        features = np.concatenate([spatial, temporal], axis=1)

        if len(features) != self.max_frames:
             indices = np.linspace(0, len(features)-1, self.max_frames, dtype=int)
             features = features[indices]
        
        x = torch.tensor(features, dtype=torch.float32)
        x = (x - self.mean) / self.std
        
        if torch.isnan(x).any() or torch.isinf(x).any(): return None
            
        return x, sample['label_idx']


# 4

In [ ]:

def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch: return None, None
    seqs, lbls = zip(*batch)
    return torch.stack(seqs), torch.tensor(lbls, dtype=torch.long)


# 5

In [ ]:

class StackedBiLSTMTransformerModel(nn.Module):
    def __init__(self, 
                 input_dim: int, 
                 num_classes: int, 
                 hidden_dim: int = 384, 
                 nhead: int = 8, 
                 num_lstm_layers: int = 2,
                 num_transformer_layers: int = 1):
        
        super().__init__()
        print(f"\nbuilding stacked_bilstm_transformer_model:")
        print(f"  input dim: {input_dim}, hidden dim: {hidden_dim}, output classes: {num_classes}")
        print(f"  lstm layers: {num_lstm_layers}, transformer layers: {num_transformer_layers}, heads: {nhead}")

        self.frame_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), 
            nn.LayerNorm(hidden_dim),
            nn.GELU(), 
            nn.Dropout(0.1),
        )

        self.positional_encoding = nn.Parameter(torch.zeros(1, 256, hidden_dim))

        lstm_dropout = 0.1 if num_lstm_layers > 1 else 0.0
        self.temporal_lstm = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim // 2,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=lstm_dropout, 
            bidirectional=True
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=nhead, 
            dim_feedforward=hidden_dim * 4,
            dropout=0.1, 
            activation="gelu",
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer, 
            num_layers=num_transformer_layers
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256), 
            nn.LayerNorm(256),
            nn.GELU(), 
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
        self._init_weights()
        print(f"  total parameters: {sum(p.numel() for p in self.parameters()):,}")


    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.bias, 0)
                nn.init.constant_(m.weight, 1.0)
        
        nn.init.normal_(self.positional_encoding, std=0.02)


    def forward(self, x: torch.Tensor, src_key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        B, T, D = x.shape
        
        features = self.frame_encoder(x)
        
        if T > self.positional_encoding.shape[1]:
             raise ValueError(f"input sequence length ({T}) exceeds max positional encoding length ({self.positional_encoding.shape[1]})")
        features = features + self.positional_encoding[:, :T, :]
        
        lstm_out, _ = self.temporal_lstm(features)
        
        transformer_out = self.transformer_encoder(
            lstm_out, 
            src_key_padding_mask=src_key_padding_mask
        )
        
        if src_key_padding_mask is not None:
            mask = ~src_key_padding_mask.unsqueeze(-1)
            masked_output = transformer_out * mask
            summed = torch.sum(masked_output, dim=1)
            count = mask.sum(dim=1).clamp(min=1e-9)
            pooled = summed / count
        else:
            pooled = torch.mean(transformer_out, dim=1) 
        
        return self.classifier(pooled)
    

# 6

In [ ]:

def sanity_check_overfit(model_class, model_args, train_loader, device, max_epochs=200):
    print(f"\nsanity check: attempting to overfit a single batch")
    model = model_class(**model_args).to(device)

    single_batch = None
    for seqs, lbls in train_loader:
        if seqs is None:
            continue
        single_batch = (seqs, lbls)
        break

    if single_batch is None:
        print("could not load a valid batch!"); return False
    
    seqs, lbls = single_batch[0].to(device), single_batch[1].to(device)
    print(f"  batch size: {seqs.shape[0]}, unique labels: {len(torch.unique(lbls))}")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(seqs)
        loss = criterion(outputs, lbls)
        if torch.isnan(loss).any().item():
            print("   nan loss detected; skipping step.")
            continue
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 20 == 0:
            acc = (outputs.argmax(1) == lbls).float().mean().item() * 100
            print(f"   epoch {epoch+1:3d}: loss={loss.item():.4f}, acc={acc:.2f}%")
            if acc > 95:
                print(f"\n   success! overfitted in {epoch+1} epochs. model can learn.")
                return True
    
    print(f"\n   failure! could not overfit.")
    return False


## 6.2

In [ ]:

def compute_comprehensive_metrics(all_preds, all_labels, num_classes, epoch, phase='Val'):
    print(f"\n")
    print(f"{phase} metrics - epoch {epoch}")
    print(f"")
    
    accuracy = accuracy_score(all_labels, all_preds)
    
    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    precision_weighted = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall_weighted = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    print(f"\noverall metrics:")
    print(f"  accuracy:         {accuracy*100:.2f}%")
    print(f"\n  macro averages:")
    print(f"  - precision:      {precision_macro*100:.2f}%")
    print(f"  - recall:         {recall_macro*100:.2f}%")
    print(f"\n  weighted averages:")
    print(f"  - precision:      {precision_weighted*100:.2f}%")
    print(f"  - recall:         {recall_weighted*100:.2f}%")
    print(f"\n  - f1-score (macro): {f1_macro*100:.2f}%")
    print(f"  - f1-score (weighted): {f1_weighted*100:.2f}%")
    
    cm = confusion_matrix(all_labels, all_preds)
    print(f"\nconfusion matrix statistics:")
    print(f"  true positives:    {np.diag(cm).sum()}")
    print(f"  total predictions:  {cm.sum()}")
    
    metrics_dict = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm
    }
    
    return metrics_dict

def plot_confusion_matrix(cm, epoch, phase='Val', save_path='confusion_matrix.png', top_k=50):
    if cm.shape[0] > top_k:
        row_sums = cm.sum(axis=1)
        top_indices = np.argsort(row_sums)[-top_k:]
        cm_subset = cm[np.ix_(top_indices, top_indices)]
    else:
        cm_subset = cm
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_subset, annot=False, fmt='d', cmap='Blues', cbar_kws={'label': 'count'})
    plt.title(f'{phase} confusion matrix - epoch {epoch}\n(showing {"top " + str(top_k) if cm.shape[0] > top_k else "all"} classes)', fontsize=14)
    plt.ylabel('true label', fontsize=12)
    plt.xlabel('predicted label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   confusion matrix saved to: {save_path}")

def plot_metrics_history(history, save_path='training_metrics.png'):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('training history', fontsize=16, fontweight='bold')
    
    metrics = [
        ('accuracy', 'accuracy', '%'),
        ('f1_macro', 'f1-score (macro)', '%'),
        ('precision_macro', 'precision (macro)', '%'),
        ('recall_macro', 'recall (macro)', '%'),
        ('loss', 'loss', ''),
        ('f1_weighted', 'f1-score (weighted)', '%')
    ]
    
    for idx, (metric, title, unit) in enumerate(metrics):
        ax = axes[idx // 3, idx % 3]
        
        train_key = f'train_{metric}'
        val_key = f'val_{metric}'
        
        if train_key in history:
            epochs = range(1, len(history[train_key]) + 1)
            train_vals = [v * 100 if unit == '%' and v <= 1 else v for v in history[train_key]]
            val_vals = [v * 100 if unit == '%' and v <= 1 else v for v in history[val_key]]
            
            ax.plot(epochs, train_vals, 'b-o', label='train', linewidth=2, markersize=4)
            ax.plot(epochs, val_vals, 'r-s', label='val', linewidth=2, markersize=4)
            ax.set_xlabel('epoch', fontsize=10)
            ax.set_ylabel(f'{title} {unit}', fontsize=10)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend(loc='best')
            ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   training history saved to: {save_path}")

def evaluate_model(model, loader, device, criterion, num_classes, epoch, phase='Val'):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    batches = 0
    
    with torch.no_grad():
        for seqs, lbls in tqdm(loader, desc=f"{phase} evaluation", ncols=100):
            if seqs is None: continue
            seqs, lbls = seqs.to(device), lbls.to(device)
            outputs = model(seqs)
            loss = criterion(outputs, lbls)
            total_loss += loss.item()
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            batches += 1
    
    avg_loss = total_loss / max(1, batches)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    metrics = compute_comprehensive_metrics(all_preds, all_labels, num_classes, epoch, phase)
    metrics['loss'] = avg_loss
    
    return metrics


# 7

In [ ]:

def train_model(model, train_loader, val_loader, device, epochs, save_path, num_classes):
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0
    best_val_f1 = 0.0
    patience_counter = 0
    patience_limit = 15
    
    history = {
        'train_loss': [], 'train_accuracy': [], 'train_f1_macro': [], 
        'train_precision_macro': [], 'train_recall_macro': [], 'train_f1_weighted': [],
        'val_loss': [], 'val_accuracy': [], 'val_f1_macro': [],
        'val_precision_macro': [], 'val_recall_macro': [], 'val_f1_weighted': []
    }

    for epoch in range(epochs):
        print(f"\n")
        print(f"epoch {epoch+1}/{epochs}")
        print(f"")
        
        model.train()
        train_preds = []
        train_labels = []
        train_loss = 0
        train_batches = 0
        
        for seqs, lbls in tqdm(train_loader, desc="training", ncols=100):
            if seqs is None: continue
            seqs, lbls = seqs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, lbls)
            if torch.isnan(loss).any().item(): 
                print("   nan loss detected during training; skipping batch.")
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            train_loss += loss.item()
            preds = outputs.argmax(1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(lbls.cpu().numpy())
            train_batches += 1
        
        train_preds = np.array(train_preds)
        train_labels = np.array(train_labels)
        train_metrics = compute_comprehensive_metrics(train_preds, train_labels, num_classes, epoch+1, 'train')
        train_metrics['loss'] = train_loss / max(1, train_batches)
        
        val_metrics = evaluate_model(model, val_loader, device, criterion, num_classes, epoch+1, 'val')
        
        for key in ['loss', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'f1_weighted']:
            history[f'train_{key}'].append(train_metrics[key])
            history[f'val_{key}'].append(val_metrics[key])
        
        print(f"\nepoch {epoch+1} summary:")
        print(f"   train -> loss: {train_metrics['loss']:.4f}, acc: {train_metrics['accuracy']*100:.2f}%, f1: {train_metrics['f1_macro']*100:.2f}%")
        print(f"   val   -> loss: {val_metrics['loss']:.4f}, acc: {val_metrics['accuracy']*100:.2f}%, f1: {val_metrics['f1_macro']*100:.2f}%")
        
        scheduler.step(val_metrics['accuracy'])
        
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_val_f1 = val_metrics['f1_macro']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch + 1,
                'val_acc': val_metrics['accuracy'],
                'val_f1': val_metrics['f1_macro'],
                'train_metrics': train_metrics,
                'val_metrics': val_metrics
            }, save_path)
            print(f"new best model saved! val acc: {val_metrics['accuracy']*100:.2f}%, val f1: {val_metrics['f1_macro']*100:.2f}%")
            
            plot_confusion_matrix(val_metrics['confusion_matrix'], epoch+1, 'val', 
                                  f'confusion_matrix_epoch_{epoch+1}.png')
        else:
            patience_counter += 1
            if patience_counter >= patience_limit:
                print("early stopping.")
                break
    
    plot_metrics_history(history, 'training_history.png')
    
    print(f"\n")
    print(f"training complete")
    print(f"")
    print(f"   best validation accuracy: {best_val_acc*100:.2f}%")
    print(f"   best validation f1-score: {best_val_f1*100:.2f}%")
    
    return best_val_acc, best_val_f1, history


# 8

In [ ]:

def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"using device: {device}")
    
    config = {
        'full_dataset_ann': r'D:\Balanced_20_Frames_Augmented\train_final.json',
        'full_dataset_root': r'D:\Balanced_20_Frames_Augmented\Train',
        'label_map': r'D:\Balanced_20_Frames_Augmented\label_map_final.json',
        'stats_file': r'D:\Balanced_20_Frames_Augmented\stats.json',
        'batch_size': 32,
        'epochs': 30,
        'top_n': 200,
        'val_split': 0.2
    }

    full_dataset = FixedLandmarkDataset(
        config['full_dataset_ann'], config['full_dataset_root'], config['label_map'], 
        config['stats_file'], top_n_classes=config['top_n']
    )
    
    print(f"\nsplitting data into {1-config['val_split']:.0%}/{config['val_split']:.0%} train/val sets...")
    dataset_size = len(full_dataset)
    val_size = int(dataset_size * config['val_split'])
    train_size = dataset_size - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    print(f"   train samples: {len(train_dataset)}, validation samples: {len(val_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], collate_fn=collate_fn, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size']*2, collate_fn=collate_fn, num_workers=0)

    model_args = {'input_dim': 3484, 'num_classes': config['top_n'], 'hidden_dim': 384}
    if not sanity_check_overfit(StackedBiLSTMTransformerModel, model_args, train_loader, device):
        print("\nsanity check failed. halting.")
        return

    print(f"\nstarting full training (30 epochs)")
    model = StackedBiLSTMTransformerModel(**model_args).to(device)
    best_acc, best_f1, history = train_model(
        model, train_loader, val_loader, device, 
        epochs=config['epochs'], save_path='final_model.pth',
        num_classes=config['top_n']
    )
    
    print(f"\nall done!")
    print(f"   best validation accuracy: {best_acc*100:.2f}%")
    print(f"   best validation f1-score: {best_f1*100:.2f}%")



In [ ]:
main()